# JAV Thumbnail Uploader Bot (Google Colab)

A standalone Telegram bot that accepts manually supplied JAV codes, searches JAVDatabase using the `javdb` Python package, downloads an exact-match poster (with preview-image fallback), and uploads it to a Telegram channel with the code as the caption.

**Run cells from top to bottom.** Add the bot as an administrator in the destination channel with permission to post photos. Keep the runtime running while processing.

In [ ]:
!pip -q install -U javdb aiogram pillow


## 1. Configuration

Paste your values directly into this cell. No Colab Secrets or environment variables are used. Do not commit real credentials to GitHub.

In [ ]:
# Telegram configuration — enter your own values
BOT_TOKEN = "PASTE_TELEGRAM_BOT_TOKEN_HERE"
CHANNEL_ID = -1001234567890  # Destination channel ID
OWNER_ID = 123456789         # Your numeric Telegram user ID

# Batch and reliability settings
REQUEST_DELAY_SECONDS = 2.0
MAX_CODES_PER_BATCH = 200
PROGRESS_UPDATE_EVERY = 5
SEARCH_PAGES = 2


## 2. Persistent duplicate protection

Google Drive is used so uploaded codes remain recorded after a Colab runtime restart. Set `USE_GOOGLE_DRIVE = False` if you prefer temporary runtime storage.

In [ ]:
import os, json

USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/JAVThumbnailBot'
else:
    DATA_DIR = '/content/JAVThumbnailBot'

os.makedirs(DATA_DIR, exist_ok=True)
PROCESSED_FILE = os.path.join(DATA_DIR, 'processed_codes.json')

if os.path.isfile(PROCESSED_FILE):
    with open(PROCESSED_FILE, 'r', encoding='utf-8') as f:
        processed_codes = set(json.load(f))
else:
    processed_codes = set()

def save_processed_codes():
    temp_path = PROCESSED_FILE + '.tmp'
    with open(temp_path, 'w', encoding='utf-8') as f:
        json.dump(sorted(processed_codes), f, ensure_ascii=False, indent=2)
    os.replace(temp_path, PROCESSED_FILE)

print(f'Loaded {len(processed_codes)} previously uploaded codes.')


## 3. JAVDatabase lookup and thumbnail downloader

The bot requires an exact code match in search results and verifies the movie detail page's `dvd_id` before downloading. It tries the poster first, then preview images.

In [ ]:
import re, tempfile, asyncio
from pathlib import Path
from PIL import Image, ImageOps
from javdb import Client, JavDBError

jav_client = Client(timeout=20, retries=2)

def normalize_code(value):
    return re.sub(r'[\s_]+', '', str(value or '')).upper().strip()

def fetch_thumbnail(code, work_dir):
    results = jav_client.search(code, page=1, pages=SEARCH_PAGES)
    target = normalize_code(code)
    exact = next((item for item in results if normalize_code(item.get('code')) == target), None)
    if not exact:
        raise ValueError('No exact code match in search results')

    movie = jav_client.movie(exact['link'])
    detail_code = normalize_code(movie.get('dvd_id'))
    if detail_code != target:
        raise ValueError(f'Exact detail-page code verification failed (received: {movie.get("dvd_id")!r})')

    image_urls = []
    if movie.get('poster'):
        image_urls.append(movie['poster'])
    image_urls.extend(movie.get('preview_images') or [])
    if not image_urls:
        raise ValueError('No poster or preview images are available')

    output_path = Path(work_dir) / 'thumbnail.jpg'
    errors = []
    for number, image_url in enumerate(image_urls, start=1):
        source_path = Path(work_dir) / f'source_{number}.img'
        try:
            jav_client.download(image_url, str(source_path), overwrite=True)
            with Image.open(source_path) as source:
                image = ImageOps.exif_transpose(source).convert('RGB')
                image.thumbnail((1600, 1600))
                image.save(output_path, format='JPEG', quality=90, optimize=True)
            if output_path.is_file() and output_path.stat().st_size > 0:
                return str(output_path)
        except Exception as exc:
            errors.append(f'image {number}: {exc}')
    raise RuntimeError('All image downloads failed: ' + '; '.join(errors[-3:]))

print('JAVDatabase client ready.')


## 4. Telegram bot and batch processor

Send codes as a message, one per line or separated by spaces/commas. You can also use `/add CODE1 CODE2`. The bot processes one batch at a time and skips codes already recorded in Google Drive.

In [ ]:
from aiogram import Bot, Dispatcher, F
from aiogram.filters import Command
from aiogram.types import Message, FSInputFile

if not BOT_TOKEN or 'PASTE_' in BOT_TOKEN:
    raise ValueError('Set BOT_TOKEN in the Configuration cell first.')
if not isinstance(CHANNEL_ID, int) or not isinstance(OWNER_ID, int):
    raise TypeError('CHANNEL_ID and OWNER_ID must be integers.')

# Self-heal if the persistence cell was skipped or the runtime was reset.
if 'processed_codes' not in globals():
    import os, json
    candidate_dir = globals().get('DATA_DIR')
    if not candidate_dir:
        drive_dir = '/content/drive/MyDrive/JAVThumbnailBot'
        candidate_dir = drive_dir if os.path.isdir('/content/drive/MyDrive') else '/content/JAVThumbnailBot'
        print('⚠️ Duplicate-tracking cell was skipped. Initializing tracking automatically.')
        if candidate_dir == '/content/JAVThumbnailBot':
            print('⚠️ Google Drive is not mounted; duplicate history will be temporary for this runtime.')
    DATA_DIR = candidate_dir
    os.makedirs(DATA_DIR, exist_ok=True)
    PROCESSED_FILE = globals().get('PROCESSED_FILE', os.path.join(DATA_DIR, 'processed_codes.json'))
    if os.path.isfile(PROCESSED_FILE):
        with open(PROCESSED_FILE, 'r', encoding='utf-8') as f:
            processed_codes = set(json.load(f))
    else:
        processed_codes = set()

    def save_processed_codes():
        temp_path = PROCESSED_FILE + '.tmp'
        with open(temp_path, 'w', encoding='utf-8') as f:
            json.dump(sorted(processed_codes), f, ensure_ascii=False, indent=2)
        os.replace(temp_path, PROCESSED_FILE)

bot = Bot(token=BOT_TOKEN)
dp = Dispatcher()
active_task = None
current_status = 'Idle'

CODE_PATTERN = re.compile(r'(?i)\b[A-Z0-9]+(?:-[A-Z0-9]+)*-\d+[A-Z0-9]*\b')

def extract_codes(text):
    found = CODE_PATTERN.findall(text or '')
    result, seen = [], set()
    for item in found:
        code = item.upper()
        key = normalize_code(code)
        if key not in seen:
            seen.add(key)
            result.append(code)
    return result

def is_owner(message):
    return bool(message.from_user and message.from_user.id == OWNER_ID)

async def process_batch(message, codes):
    global current_status, active_task
    total = len(codes)
    uploaded = skipped = 0
    failures = []
    progress = await message.answer(f'🚀 Started: {total} code(s).')
    try:
        for index, code in enumerate(codes, start=1):
            key = normalize_code(code)
            current_status = f'Processing {index}/{total}: {code} | Uploaded {uploaded} | Skipped {skipped} | Failed {len(failures)}'
            if key in processed_codes:
                skipped += 1
            else:
                try:
                    with tempfile.TemporaryDirectory(prefix='javthumb_') as folder:
                        image_path = await asyncio.to_thread(fetch_thumbnail, code, folder)
                        await bot.send_photo(
                            chat_id=CHANNEL_ID,
                            photo=FSInputFile(image_path),
                            caption=code
                        )
                    processed_codes.add(key)
                    save_processed_codes()
                    uploaded += 1
                except asyncio.CancelledError:
                    raise
                except Exception as exc:
                    failures.append(f'{code}: {str(exc)[:180]}')

            if index % PROGRESS_UPDATE_EVERY == 0 or index == total:
                current_status = f'{index}/{total} | Uploaded {uploaded} | Skipped {skipped} | Failed {len(failures)}'
                try:
                    await progress.edit_text('⏳ ' + current_status)
                except Exception:
                    pass
            if index < total:
                await asyncio.sleep(REQUEST_DELAY_SECONDS)

        report = (f'✅ Batch finished\n\n📦 Total: {total}\n📤 Uploaded: {uploaded}'
                  f'\n⏭ Already uploaded: {skipped}\n❌ Failed: {len(failures)}')
        if failures:
            report += '\n\nFailed codes:\n' + '\n'.join(failures[:30])
        await message.answer(report)
    except asyncio.CancelledError:
        await message.answer(f'🛑 Batch cancelled. Uploaded: {uploaded}; skipped: {skipped}; failed: {len(failures)}.')
        raise
    except Exception as exc:
        current_status = f'Batch stopped: {type(exc).__name__}: {str(exc)[:160]}'
        try:
            await message.answer('❌ Batch stopped unexpectedly: ' + current_status)
        except Exception:
            pass
    finally:
        current_status = 'Idle'
        active_task = None

async def start_batch(message, codes):
    global active_task
    if active_task and not active_task.done():
        await message.answer('⚠️ A batch is already running. Use /status or /cancel.')
        return
    active_task = asyncio.create_task(process_batch(message, codes))

@dp.message(Command('start'))
async def start_handler(message: Message):
    if not is_owner(message):
        return
    await message.answer('🎬 JAV Thumbnail Uploader\n\nSend codes separated by new lines, spaces, or commas.\n\n/add CODE1 CODE2 — add codes\n/status — progress\n/cancel — stop current batch')

@dp.message(Command('add'))
async def add_handler(message: Message):
    if not is_owner(message):
        return
    raw = (message.text or '').partition(' ')[2]
    codes = extract_codes(raw)
    if not codes:
        await message.answer('Send codes after /add, for example: /add SONE-763 IPZZ-979')
        return
    if len(codes) > MAX_CODES_PER_BATCH:
        await message.answer(f'Too many codes. Maximum per batch: {MAX_CODES_PER_BATCH}.')
        return
    await start_batch(message, codes)

@dp.message(Command('status'))
async def status_handler(message: Message):
    if is_owner(message):
        await message.answer('📊 ' + current_status)

@dp.message(Command('cancel'))
async def cancel_handler(message: Message):
    if not is_owner(message):
        return
    if active_task and not active_task.done():
        active_task.cancel()
        await message.answer('🛑 Cancellation requested.')
    else:
        await message.answer('No batch is running.')

@dp.message(F.text)
async def text_handler(message: Message):
    if not is_owner(message):
        return
    codes = extract_codes(message.text or '')
    if not codes:
        await message.answer('No supported JAV codes found in that message.')
        return
    if len(codes) > MAX_CODES_PER_BATCH:
        await message.answer(f'Too many codes. Maximum per batch: {MAX_CODES_PER_BATCH}.')
        return
    await start_batch(message, codes)

async def run_bot():
    print('JAV Thumbnail Bot is running. Keep this cell active.')
    await dp.start_polling(bot, allowed_updates=dp.resolve_used_update_types())

print('Bot handlers registered.')


## 5. Start the bot

Run this cell last. Keep it running while you use the bot in Telegram.

In [ ]:
await run_bot()
